In [ ]:
!pip install paddleocr
!pip install paddlepaddle
!pip install paddle


In [ ]:
# قدم سوم: اجرای کد اصلی بعد از ری‌استارت
from paddleocr import PaddleOCR
import re

# استفاده از پارامتر جدید به جای پارامتر قدیمی
ocr = PaddleOCR(lang='en', use_textline_orientation=True)

img_path = '/content/566w-7cbCXZOFDC8.jpg'

try:
    results = ocr.ocr(img_path, cls=True) # استفاده از متد کامل ocr
    for idx in range(len(results)):
        res = results[idx]
        for line in res:
            print(line[1][0]) # چاپ متن شناسایی شده
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
from paddleocr import PaddleOCR
import json
import re

ocr = PaddleOCR(lang='en', use_angle_cls=True)
img_path = '/content/566w-7cbCXZOFDC8.jpg'
results = ocr.predict(img_path)

lines = results[0]["rec_texts"]

def extract_next_line(keywords, lines):
    for i, line in enumerate(lines):
        for kw in keywords:
            if kw.lower() in line.lower():
                if i + 1 < len(lines):
                    return lines[i+1].strip()
    return None

def extract_same_line(keywords, lines, pattern=None):
    for line in lines:
        for kw in keywords:
            if kw.lower() in line.lower():
                if pattern:
                    m = re.search(pattern, line, re.I)
                    if m:
                        return m.group(1).strip()
                else:
                    return line.split(kw)[-1].strip()
    return None

invoice_data = {
    "invoice_no": extract_next_line(["invoice no"], lines),
    "date":       extract_same_line(["date"], lines, r"date[: ]+([\d./-]+)"),
    "due_date":   extract_same_line(["due date"], lines, r"due\s*date[: ]+([\d./-]+)"),
    "subtotal":   extract_next_line(["subtotal"], lines),
    "tax":        extract_next_line(["tax"], lines),
    "total":      extract_next_line(["total"], lines),
    "issued_to":  extract_next_line(["issued to"], lines),
}

# === استخراج آیتم‌های جدول با گروه‌بندی ===
items = []
in_table = False
row_buffer = []

for line in lines:
    if "description" in line.lower():
        in_table = True
        continue
    if "subtotal" in line.lower():
        in_table = False
        continue

    if in_table:
        row_buffer.append(line.strip())

        # هر 4 خط یک آیتم (desc, unit, qty, total)
        if len(row_buffer) == 4:
            desc, unit_price, qty, total = row_buffer
            items.append({
                "description": desc,
                "unit_price": unit_price,
                "qty": qty,
                "total": total
            })
            row_buffer = []

invoice_data["items"] = items

print("\n=== INVOICE DATA (JSON) ===")
print(json.dumps(invoice_data, indent=2))

